# Per-Subject CF Modeling — Results Visualization

Pycortex flatmaps for group-average statistics from scripts 05 and 06.
Use wb_view for CIFTI visualization (59k and 32k files in `group/cifti_59k/` and `group/cifti_32k/`).

**Before running:** confirm `cortex.database.default_filestore` points to your HCP filestore
and that subject `hcp_999999_draw_NH` is registered.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import cortex

# Make shared/ and vendor/ importable
CF_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if CF_DIR not in sys.path:
    sys.path.insert(0, CF_DIR)

from shared.vis_utils import basic_plot, alpha_plot, Plot

In [ ]:
# =============================================================================
# CONFIG — edit these
# =============================================================================
ROI_A = "A5"
ROI_B = "FFC"

OUTPUT_BASE = "/home/amin/Research/Representation/Movie/outputs/cf_modeling/per_subject"
HCP_DIR     = "/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1"
CX_SUB      = "hcp_999999_draw_NH"

OUTPUT_ROOT = f"{OUTPUT_BASE}/{ROI_A}_{ROI_B}"
GROUP_DIR   = f"{OUTPUT_ROOT}/group"
STATS_DIR   = f"{GROUP_DIR}/stats"
FIGURES_DIR = f"{GROUP_DIR}/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

cortex.database.default_filestore = HCP_DIR
print(f"ROI pair: {ROI_A} × {ROI_B}")
print(f"Group dir: {GROUP_DIR}")

In [ ]:
# Load group-average null-corrected R² maps (32k, 59412 vertices)
R2_a_nc = np.load(f"{GROUP_DIR}/R2_{ROI_A}_nc_avg.npy")
R2_b_nc = np.load(f"{GROUP_DIR}/R2_{ROI_B}_nc_avg.npy")

# Integration and balance maps
integration = np.load(f"{GROUP_DIR}/integration_score_avg.npy")
balance     = np.load(f"{GROUP_DIR}/modality_balance_avg.npy")

# Cohen's d maps (from script 06)
d_a = np.load(f"{STATS_DIR}/d_{ROI_A}_nc.npy")
d_b = np.load(f"{STATS_DIR}/d_{ROI_B}_nc.npy")
fdr_a = np.load(f"{STATS_DIR}/fdr_mask_{ROI_A}_nc.npy").astype(bool)
fdr_b = np.load(f"{STATS_DIR}/fdr_mask_{ROI_B}_nc.npy").astype(bool)

print(f"R2_{ROI_A}_nc: mean={R2_a_nc.mean():.4f}  frac>0={np.mean(R2_a_nc>0):.1%}")
print(f"R2_{ROI_B}_nc: mean={R2_b_nc.mean():.4f}  frac>0={np.mean(R2_b_nc>0):.1%}")
print(f"FDR-sig {ROI_A}: {fdr_a.sum()} verts | {ROI_B}: {fdr_b.sum()} verts")

## 1. Group-average R² maps (null-corrected)

In [ ]:
vmax_a = np.percentile(R2_a_nc[R2_a_nc > 0], 95) if (R2_a_nc > 0).any() else 0.1
vmax_b = np.percentile(R2_b_nc[R2_b_nc > 0], 95) if (R2_b_nc > 0).any() else 0.1

fig_a = basic_plot(R2_a_nc, vmax=vmax_a, subject=CX_SUB, vmin=0, cmap="Reds")
fig_a.suptitle(f"Group R² {ROI_A} (null-corrected)")

fig_b = basic_plot(R2_b_nc, vmax=vmax_b, subject=CX_SUB, vmin=0, cmap="Blues")
fig_b.suptitle(f"Group R² {ROI_B} (null-corrected)")

## 2. Variance decomposition — 2D dual-colormap (Figure 3a style)

Red = ROI_A only, Blue = ROI_B only, Purple = bimodal integration zone

In [ ]:
p = Plot(
    dat=R2_a_nc,
    dat2=R2_b_nc,
    vmin=0, vmax=vmax_a,
    vmin2=0, vmax2=vmax_b,
    subject=CX_SUB,
    with_curvature=True,
    with_rois=True,
)
p.uber_plot()
p.fig.suptitle(f"Variance decomposition: {ROI_A} (red) × {ROI_B} (blue)")

out_path = f"{FIGURES_DIR}/variance_decomp_2D.png"
p.saveout(outloc=out_path)
print(f"Saved: {out_path}")

## 3. Integration score (geometric mean)

In [ ]:
vmax_int = np.percentile(integration[integration > 0], 95) if (integration > 0).any() else 0.05
fig_int = basic_plot(integration, vmax=vmax_int, subject=CX_SUB, vmin=0, cmap="viridis")
fig_int.suptitle(f"Integration score √(R²_{ROI_A} × R²_{ROI_B})")

## 4. Cohen's d maps

In [ ]:
vmax_d = max(
    np.percentile(d_a[d_a > 0], 95) if (d_a > 0).any() else 1.0,
    np.percentile(d_b[d_b > 0], 95) if (d_b > 0).any() else 1.0,
)

fig_da = basic_plot(d_a, vmax=vmax_d, subject=CX_SUB, vmin=0, cmap="hot")
fig_da.suptitle(f"Cohen's d — {ROI_A} effect")

fig_db = basic_plot(d_b, vmax=vmax_d, subject=CX_SUB, vmin=0, cmap="Blues")
fig_db.suptitle(f"Cohen's d — {ROI_B} effect")

## 5. FDR-thresholded Cohen's d maps

In [ ]:
d_a_fdr = d_a.copy(); d_a_fdr[~fdr_a] = 0
d_b_fdr = d_b.copy(); d_b_fdr[~fdr_b] = 0

fig_da_fdr = basic_plot(d_a_fdr, vmax=vmax_d, subject=CX_SUB, vmin=0, cmap="hot")
fig_da_fdr.suptitle(f"Cohen's d — {ROI_A} (FDR q<0.05)")

fig_db_fdr = basic_plot(d_b_fdr, vmax=vmax_d, subject=CX_SUB, vmin=0, cmap="Blues")
fig_db_fdr.suptitle(f"Cohen's d — {ROI_B} (FDR q<0.05)")

## 6. Save flatmaps to PNG

In [ ]:
def save_flatmap(data, vmax, cmap, name, subject=CX_SUB):
    vx  = cortex.Vertex(data, subject=subject, vmin=0, vmax=vmax, cmap=cmap)
    fig = cortex.quickflat.make_figure(vx, with_curvature=True, with_rois=True)
    out = f"{FIGURES_DIR}/{name}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")

save_flatmap(d_a,     vmax_d, "hot",    f"d_{ROI_A}_nc_flatmap")
save_flatmap(d_b,     vmax_d, "Blues",  f"d_{ROI_B}_nc_flatmap")
save_flatmap(d_a_fdr, vmax_d, "hot",    f"d_{ROI_A}_nc_fdr_flatmap")
save_flatmap(d_b_fdr, vmax_d, "Blues",  f"d_{ROI_B}_nc_fdr_flatmap")
save_flatmap(integration, vmax_int, "viridis", "integration_score_flatmap")

---
## wb_view: CIFTI visualization

Use wb_view for vertex-level inspection of the 59k/32k CIFTI files:

```bash
# 59k native space
wb_view $GROUP_DIR/cifti_59k/R2_{ROI_A}_nc_avg.dscalar.nii

# 32k resampled (same space as group_average pipeline)
wb_view $GROUP_DIR/cifti_32k/R2_{ROI_A}_nc_avg.dscalar.nii

# Bimodal label map
wb_view $GROUP_DIR/cifti_59k/bimodal_map.dlabel.nii
```